<a href="https://colab.research.google.com/github/3132Ganesh/wtsapp_group_dna/blob/Spend_DNA/Minorproject_2_ganeshnayak.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# spend DNA
**wallet's year end story**

In [ ]:
#data set parsing
import pandas as pd
import numpy as np

df = pd.read_csv("Data set for DADS June.csv")

# Clean Amount
def parse_amount(x):
    if pd.isna(x): return np.nan
    x = str(x).replace("₹","").replace("Rs.","").replace(",","").strip()
    try: return float(x)
    except: return np.nan

df["Amount_Clean"] = df["Amount"].apply(parse_amount)

# Normalize Type
df["Txn_Type"] = df["Type"].str.upper().map({"DR":"Debit","DEBIT":"Debit","CR":"Credit"})

# Parse datetime
df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"], errors="coerce")
df

In [ ]:
print(df['Description'].unique()[:50])

In [ ]:
vendor_dict = {
    "Swiggy": ["SWIGGY", "INSTAMART", "BUNDL"],
    "Zomato": ["ZOMATO"],
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA"],
    "Dmart": ["DMART"],
    "BigBasket": ["BIGBASKET"],
    "Blinkit": ["BLINKIT", "GROFERS"],
    "Uber": ["UBER"],
    "Ola": ["OLA"],
    "Rapido": ["RAPIDO"],
    "BMTC": ["BMTC", "TUMMOC"],
    "Starbucks": ["STARBUCKS"],
    "CCD": ["COFFEE DAY", "CCD"],
    "Truffles": ["TRUFFLES"],
    "Empire": ["EMPIRE RESTAURANT"],
    "Dineout": ["DINEOUT"],
    "Petrol": ["HP PETROL", "INDIAN OIL", "BPCL"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Hotstar": ["HOTSTAR", "STAR INDIA"],
    "Airtel": ["AIRTEL"],
    "Jio": ["JIO", "RELIANCE JIO"],
    "Cash Withdrawal": ["ATM-WDL"],
    "P2P Transfer": ["UPI-PRIYA", "UPI-NEHA", "UPI-AMAN", "UPI-VIKAS"],  # friend transfers
    "Salary": ["SALARY"],
    "Rent": ["RENT", "LANDLORD"],
    "Zerodha": ["ZERODHA"],
    "Groww": ["GROWWPAY"],
    "FSN Ecommerce": ["FSN E-COMMERCE"],  # Nykaa parent
    "Innovative Retail": ["INNOVATIVE RETAIL"],  # BigBasket parent
}
def extract_vendor(desc):
    desc_upper = str(desc).upper()
    for vendor, keywords in vendor_dict.items():
        for kw in keywords:
            if kw in desc_upper:
                return vendor
    return "Other"

df["vendor_clean"] = df["Description"].apply(extract_vendor)
print("Unique vendors:", df['vendor_clean'].nunique())
print(df['vendor_clean'].value_counts().head(10))


In [ ]:
category_dict = {
    # Food Delivery
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick Commerce
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",
    "BigBasket": "Quick Commerce",
    "Grofers": "Quick Commerce",

    # E-commerce
    "Amazon": "Ecommerce",
    "Flipkart": "Ecommerce",
    "Myntra": "Ecommerce",
    "Nykaa": "Ecommerce",
    "FSN Ecommerce": "Ecommerce",

    # Transport
    "Ola": "Transport",
    "Uber": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    # Cafe
    "Starbucks": "Cafe",
    "CCD": "Cafe",
    "Third Wave Coffee": "Cafe",

    # Restaurants
    "Truffles": "Restaurants",
    "Empire": "Restaurants",
    "Dineout": "Restaurants",
    "Meghana Foods": "Restaurants",

    # Subscriptions
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Hotstar": "Subscriptions",
    "Amazon Prime Video": "Subscriptions",

    # Utilities
    "BESCOM": "Utilities",
    "Jio": "Utilities",
    "Airtel": "Utilities",

    # Groceries
    "Dmart": "Groceries",
    "Avenue Supermarts": "Groceries",
    "Innovative Retail": "Groceries",

    # Investments
    "Zerodha": "Investments",
    "Groww": "Investments",

    # Fuel
    "Petrol": "Fuel",
    "Indian Oil": "Fuel",
    "BPCL": "Fuel",

    # Entertainment
    "BookMyShow": "Entertainment",
    "Disney Hotstar": "Entertainment",

    # Personal Transfer
    "P2P Transfer": "Personal Transfer",

    # Cash Withdrawal
    "Cash Withdrawal": "Cash Withdrawal",

    # Salary / Rent
    "Salary": "Income",
    "Rent": "Housing"
}
df["category"] = df["vendor_clean"].map(category_dict).fillna("Other")
print(df["category"].value_counts())


In [ ]:
# Total credits (money in)
total_credits = df[df["Txn_Type"]=="Credit"]["Amount_Clean"].sum()

# Total debits (money out)
total_debits = df[df["Txn_Type"]=="Debit"]["Amount_Clean"].sum()

# Net savings
net_savings = total_credits - total_debits

# Savings rate (% of credits retained after debits)
savings_rate = (net_savings / total_credits) * 100

# Top 5 categories by spend
top_categories = df.groupby("category")["Amount_Clean"].sum().sort_values(ascending=False).head(5)

# Top 5 vendors by spend
top_vendors = df.groupby("vendor_clean")["Amount_Clean"].sum().sort_values(ascending=False).head(5)

# Total transaction count
txn_count = len(df)
print("===========================================================")
print("   Spending Overview – Executive Summary")
print("===========================================================")
print(f"Total Credits : ₹{total_credits:,.0f}")
print(f"Total Debits  : ₹{total_debits:,.0f}")
print(f"Net Savings   : ₹{net_savings:,.0f}")
print(f"Savings Rate  : {savings_rate:.1f}%")
print(f"Total Txns    : {txn_count}")
print("\nTop Categories:\n", top_categories)
print("\nTop Vendors:\n", top_vendors)
print("===========================================================")


In [ ]:
df["Month"] = df["DateTime"].dt.month_name().str[:3]  # Jan, Feb, Mar...
month_pivot = df.pivot_table(
    values="Amount_Clean",
    index="category",
    columns="Month",
    aggfunc="sum",
    fill_value=0
)

print(month_pivot)


growth = {}
for cat in month_pivot.index:
    first = month_pivot.loc[cat].iloc[0]
    last = month_pivot.loc[cat].iloc[-1]
    if first > 0:
        pct_change = ((last - first) / first) * 100
    else:
        pct_change = np.nan
    growth[cat] = pct_change

growth_series = pd.Series(growth).sort_values(ascending=False)
biggest_growth = growth_series.idxmax(), growth_series.max()
biggest_decline = growth_series.idxmin(), growth_series.min()

print("Trending Up:", biggest_growth)
print("Trending Down:", biggest_decline)


In [ ]:
df["Hour"] = df["Time"].str[:2].astype(int)
# Group by category and hour
heatmap = df.groupby(["category","Hour"])["Amount_Clean"].sum().unstack(fill_value=0)

print("CATEGORY × HOUR SPEND MATRIX")
print(heatmap)
heatmap_pct = heatmap.div(heatmap.sum(axis=1), axis=0) * 100
# Example: Swiggy late-night orders
swiggy_pattern = heatmap_pct.loc["Food Delivery", 21:1].sum()  # 9 PM–1 AM
print(f"Swiggy late-night orders: {swiggy_pattern:.1f}%")

# Example: Cafe morning runs
cafe_pattern = heatmap_pct.loc["Cafe", 8:11].sum()
print(f"Cafe morning spend: {cafe_pattern:.1f}%")



In [ ]:
# Mean and std for each category
df["cat_mean"] = df.groupby("category")["Amount_Clean"].transform("mean")
df["cat_std"]  = df.groupby("category")["Amount_Clean"].transform("std")
df["z_score"] = (df["Amount_Clean"] - df["cat_mean"]) / df["cat_std"]
anomalies = df[df["z_score"] > 2].sort_values("z_score", ascending=False)
print("Top 5 Anomalies:")
print(anomalies[["Date","vendor_clean","category","Amount_Clean","z_score"]].head(5))


In [ ]:
def detect_archetypes(df):
    archetypes = []

    # Foodie-> >30% spend on Food Delivery
    food_share = df[df["category"]=="Food Delivery"]["Amount_Clean"].sum() / df[df["Txn_Type"]=="Debit"]["Amount_Clean"].sum()
    if food_share > 0.3:
        archetypes.append(("The Foodie", f"{food_share*100:.1f}% of spend on Food Delivery"))

    # Investor -> spends on Zerodha/Groww
    if df["vendor_clean"].isin(["Zerodha","Groww"]).any():
        invest_total = df[df["vendor_clean"].isin(["Zerodha","Groww"])]["Amount_Clean"].sum()
        archetypes.append(("The Investor", f"₹{invest_total:,.0f} on investments"))

    # Late-Night Snacker -> >40% of Food Delivery between 9 PM–1 AM
    df["Hour"] = df["Time"].str[:2].astype(int)
    late_night = df[(df["category"]=="Food Delivery") & (df["Hour"].isin([21,22,23,0,1]))]["Amount_Clean"].sum()
    total_food = df[df["category"]=="Food Delivery"]["Amount_Clean"].sum()
    if total_food > 0 and late_night/total_food > 0.4:
        archetypes.append(("The Late-Night Snacker", f"{late_night/total_food*100:.1f}% of food spend after 9 PM"))

    # Shopaholic -> anomalies in Ecommerce (Myntra, Amazon, Flipkart)
    ecommerce_anomalies = df[(df["category"]=="Ecommerce") & (df["z_score"]>2)]
    if not ecommerce_anomalies.empty:
        archetypes.append(("The Shopaholic", f"{len(ecommerce_anomalies)} ecommerce anomalies flagged"))

    # Streamer -> recurring subscription spends (Netflix, Spotify, Hotstar)
    subs_total = df[df["category"]=="Subscriptions"]["Amount_Clean"].sum()
    if subs_total > 0:
        archetypes.append(("The Streamer", f"₹{subs_total:,.0f} on subscriptions"))

    # Cash Heavy -> ATM withdrawals present
    if "Cash Withdrawal" in df["category"].values:
        cash_total = df[df["category"]=="Cash Withdrawal"]["Amount_Clean"].sum()
        archetypes.append(("The Cash Heavy", f"₹{cash_total:,.0f} withdrawn in cash"))

    return archetypes
archetypes = detect_archetypes(df)

print("Spending Archetypes:")
for name, metric in archetypes:
    print(f"- {name}: {metric}")


In [ ]:
# ====== FEATURE 2: Vendor Extractor ======
print("Feature 2 – Vendor Extractor")
print("Unique vendors:", df['vendor_clean'].nunique())
print(df['vendor_clean'].value_counts().head(10))
print("-----------------------------------------------------------")

# ====== FEATURE 3: Category Tagger ======
print("Feature 3 – Category Tagger")
print(df['category'].value_counts())
print("-----------------------------------------------------------")

# ====== FEATURE 4: Spending Overview ======
print("Feature 4 – Spending Overview")
print(f"Total Credits : ₹{total_credits:,.0f}")
print(f"Total Debits  : ₹{total_debits:,.0f}")
print(f"Net Savings   : ₹{net_savings:,.0f}")
print(f"Savings Rate  : {savings_rate:.1f}%")
print(f"Total Txns    : {txn_count}")
print("\nTop Categories:\n", top_categories)
print("\nTop Vendors:\n", top_vendors)
print("-----------------------------------------------------------")

# ====== FEATURE 5: Monthly Trend Analysis ======
print("Feature 5 – Monthly Trend Analysis")
print(month_pivot)
print(f"\nTrending Up   : {biggest_growth[0]} ({biggest_growth[1]:.1f}%)")
print(f"Trending Down : {biggest_decline[0]} ({biggest_decline[1]:.1f}%)")
print("-----------------------------------------------------------")

# ====== FEATURE 6: Time-of-Day Analysis ======
print("Feature 6 – Time-of-Day Analysis")
print(heatmap_pct)
print(f"\nSwiggy late-night orders: {swiggy_pattern:.1f}%")
print(f"Cafe morning spend: {cafe_pattern:.1f}%")
print("-----------------------------------------------------------")

# ====== FEATURE 7: Anomaly Detection ======
print("Feature 7 – Anomaly Detection")
print(anomalies[["Date","vendor_clean","category","Amount_Clean","z_score"]].head(10))
print(f"\nTotal anomalies flagged: {len(anomalies)}")
print("-----------------------------------------------------------")

# ====== FEATURE 8: Archetype Detection ======
print("Feature 8 – Archetype Detection")
for name, metric in archetypes:
    print(f"- {name}: {metric}")
print("-----------------------------------------------------------")
